[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeoLarryLai/goSPL-LandLab-Bridge/blob/main/goSPL-LandLab-Bridge_1_compare_SPL_vs_HyLands.ipynb)


# Compare goSPL SPL vs goSPL SPL + Landlab NLD + HyLands

Same flat domain, same timesteps, same forcing:

- **Initial topo:** flat **z = 0** plus **±1 m** elevation noise (seed 0, shared A/B)
- **Mesh:** **60×40 km**, **dx = 0.2 km**; starts from a lattice, then **jitters interior x/y** (default **0.2·dx**, seed 0). Run B: linear push elev to a **regular raster** (notebook `raster_scale=2`); DDTD + landslide stack both on that raster (**Δz + Δsoil** write-back for the whole Landlab block).
- **Left 0–30 km:** constant uplift **U = 1 mm/yr** only (no σ mixed in)
- **Right x > 30 km:** exponential subsidence **σ = σ₀·exp(−α·x_b)** only (σ₀ = **0.7 mm/yr** at the front; α = 4/L_B so far end near zero). Not live flexure.
- Shared map: `gospl_landlab.mesh_builder` → `compare_flat_tect.npz`
- **Run length / step:** set `END_YR` and `DT_YR` in the imports cell (default **4 Myr**, **5 kyr**; `tout = strat = DT_YR`). Progress preview via combo frames at `PROGRESS_TIMES_YR`; combo GIF every `PLOT_INTERVAL_YR` from saved PNGs.
- **Precipitation:** `uniform: 1.0` m/yr in the YAML
- Shared **K ≈ 5×10⁻⁶**, **D (hillslopeKa) = 0.5** (Run A diffusion)
- **Quiet runs:** `verbose=False` (default) shows ASCII tqdm only; `verbose=True` keeps stock goSPL text. `load_model` clears the output dir; F-PIT patch applied.

| Run | Fluvial | Hillslope / landslides |
|-----|---------|-------------------------|
| **A** | goSPL simple SPL | goSPL linear diffusion |
| **B** | goSPL SPL (same as A through SPL; **no SPACE**) | linear push to raster → DDTD on raster (D=0.5) → D8 FA → `prepare_hylands_routing` (MFD+DepressionFinder) → `BedrockLandslider` only → **Δz + Δsoil** write-back → goSPL FA → sed/sea; notebook `raster_scale=2` |

Quality gate: progress preview via combo frames at `PROGRESS_TIMES_YR`; combo GIF every `PLOT_INTERVAL_YR` from saved PNGs. No grain-size facies.



## 1. Paths and imports

Use the `gospl-landlab` conda env (kernel: **Python (gospl-landlab)**). The VTK shim must run **before** importing goSPL on machines where `vtk` SIGKILLs.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "gospl_landlab").exists():
    # notebook opened from notebooks/
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")

from gospl_landlab import (
    install_vtk_shim,
    BridgeConfig,
    BridgeRunner,
    load_model,
    format_run_done,
    destroy_model,
    landlab_substeps_for_dt,
    set_blas_threads,
)
from gospl_landlab.mesh_builder import build_regular_flat_mesh
from gospl_landlab.postprocess import (
    make_progress_and_gifs,
    write_combo_frames,
    stitch_combo_mp4,
    list_combo_frame_pngs,
    latest_run_dir,
    list_gospl_steps,
    pick_progress_steps,
    load_gospl_step,
    plot_compare_maps,
    mid_transect_indices,
)

install_vtk_shim(force=True)

# BLAS/OpenMP threads for numpy Landlab work (not HyLands scar Python loop)
LANDLAB_BLAS_THREADS = 10
set_blas_threads(LANDLAB_BLAS_THREADS)

import numpy as np
import matplotlib.pyplot as plt
import yaml

%matplotlib inline

# --- timing knobs (edit these) ---
END_YR = 4_000_000.0       # total run length (yr)
DT_YR = 5_000.0            # goSPL step = HDF5 / strata (yr); 5 kyr vs t_LS=20 kyr
PLOT_INTERVAL_YR = 10_000.0  # GIF frame spacing (sim may be finer)
LANDLAB_SUB_YR = 5_000.0  # target Landlab inner step for Run B (yr)
PROGRESS_TIMES_YR = (0.5e6, 2.0e6, 4.0e6)  # progress panel times
VERBOSE = False            # True = stock goSPL text; False = ASCII tqdm only
# Raster DDTD+FA+landslider stay float64 (Cython FA). Flag kept for API; ignored on raster path.
LANDLAB_FLOAT32 = False
LANDSLIDES_RETURN_YR = 10_000.0  # HyLands t_LS (yr); YAML stock 20_000; ~39%/step vs DT=5 kyr
COHESION_EFF = 1000.0  # Pa; tutorial-weak rock (YAML stock 15000)

print("ROOT =", ROOT)
print(
    f"timing: {END_YR/1e6:g} Myr · dt={DT_YR/1e3:g} kyr · plot every {PLOT_INTERVAL_YR/1e3:g} kyr · "
    f"Run B substeps={landlab_substeps_for_dt(DT_YR, target_sub_yr=LANDLAB_SUB_YR)}"
)




## 2. Build shared mesh + tectonics map

Starts from a square lattice of nodes + Delaunay triangles for goSPL, then **jitters interior x/y** (default **0.2·dx**, seed 0) and adds **±1 m** elevation noise (seed 0, shared A/B). Same shared file for Runs A and B. Run B: linear push elev to a regular raster (notebook `raster_scale=2`); DDTD + landslide stack run on that raster (**Δz + Δsoil** write-back). Mesh xy jitter is for goSPL only.

Initial topo: **z = 0** plus that ±1 m noise. Writes an imposed left-**30 km** mountain **U** + right-**30 km** exponential basin **σ** rock-uplift map (`*_tect.npz`) on a **60×40 km** domain (**0.2 km** spacing); demo YAMLs do not enable goSPL `flexure:`.



In [ ]:
mesh_prefix = ROOT / "examples" / "inputs" / "compare_flat"
mesh_path, tect_path = build_regular_flat_mesh(
    mesh_prefix,
    nx=301,
    ny=201,
    dx=200.0,
    z0=0.0,
    mountain_width_m=30_000.0,
    uplift_left_m_per_yr=0.001,  # left 0–30 km: U = 1 mm/yr only
    sigma0_m_per_yr=0.0007,      # right x>30 km: σ0 at front (0.7 mm/yr)
    alpha_per_m=4.0 / 30_000.0,  # L_B=30 km; far end near zero (not zero)
    xy_jitter_frac=0.2,
    elev_noise_m=1.0,
    seed=0,
)
data = np.load(mesh_path)
u = np.load(tect_path)["z"]
# Mesh xy is jittered; tectonics u was assigned on the *lattice* x (same node order).
nx = int(np.asarray(data["nx"]).item())
ny = int(np.asarray(data["ny"]).item())
dx = float(np.asarray(data["dx"]).item())
x_lat = np.tile(np.arange(nx) * dx, ny)
z = data["z"]
left, right = x_lat <= 30_000.0, x_lat > 30_000.0
print(mesh_path, "nodes", x_lat.size)
print(f"z flat: min={float(z.min()):g} max={float(z.max()):g} m")
print(f"left  (lattice x≤30 km): U = {float(u[left].mean())*1e3:g} mm/yr  (n={int(left.sum())})")
# Basin: report σ magnitude front → far (not −max(u) which mixes signs if mis-masked)
print(
    f"right (lattice x>30 km): σ = {-float(u[right].min())*1e3:g} … {-float(u[right].max())*1e3:g} mm/yr  "
    f"(n={int(right.sum())})"
)

# 1-D check on lattice x (jitter would zigzag +U into the basin band)
xu = np.unique(x_lat)
prof = np.array([u[x_lat == xi].mean() for xi in xu])
fig, ax = plt.subplots(figsize=(8, 2.8))
ax.plot(xu / 1000.0, prof * 1000.0, "k-", lw=1.5)
ax.axvline(30.0, color="gray", ls="--", lw=0.8)
ax.axhline(0.0, color="0.7", lw=0.5)
ax.set_xlabel("x (km)  [lattice; mesh xy is jittered separately]")
ax.set_ylabel("rock uplift rate (mm/yr)")
ax.set_title("Imposed tectonics: +U left · −σ(x) right only")
fig.tight_layout()
plt.show()



## 3. Run A — goSPL SPL + linear diffusion

Stock goSPL laws via `BridgeRunner` + default `BridgeConfig` (same quiet tqdm path as Run B when `VERBOSE=False`).


In [ ]:
# yml_a = ROOT / "examples" / "runA_gospl_spl.yml"
# m_a = load_model(yml_a, verbose=VERBOSE, dt=DT_YR, end=END_YR)
# # stock goSPL laws via BridgeRunner (same tqdm path as Run B)
# cfg_a = BridgeConfig()  # gospl.spl + gospl.linear_diffusion
# BridgeRunner(m_a, cfg_a).run()
# print(format_run_done(m_a, label="Run A (goSPL SPL+diffusion)"))
# destroy_model(m_a, verbose=VERBOSE)


## 4. Run B — goSPL SPL + Landlab NLD + HyLands via bridge

`BridgeRunner` keeps goSPL sediment/strata (flexure off in demo YAMLs; tectonics from the imposed U/σ map). Each step: goSPL FA → goSPL SPL (same as A through SPL; **no SPACE**) → linear push elev to **raster** → Landlab `DepthDependentTaylorDiffuser` on raster (inner substeps) → Landlab D8 FA → **`prepare_hylands_routing`** (Landlab `FlowDirectorMFD` → real `hill_flow__*`; `DepressionFinderAndRouter` → `flood_status_code`) → `BedrockLandslider` only → write-back **Δz + Δsoil** (`z_gospl += pull(z_raster_after − z_raster_before)`; soil depth change after DDTD+landslider remapped like elev into goSPL `Lsoil`) → goSPL FA → sed/sea → t+=dt. No irregular Voronoi for DDTD. No fake 1-neighbor `hill_flow` seed. Hillslope creep uses **D = 0.5** (soil_transport_velocity × soil_transport_decay_depth, matching Run A `hillslopeKa`); no goSPL hillslope. Notebook sets `cfg.raster_scale = 2.0` (API default 1.0; `dx_raster = dx_mesh / scale`) and `cfg.substeps` via `landlab_substeps_for_dt` (at `DT_YR=5` kyr → 1). The notebook overrides HyLands `landslides_return_time` (`t_LS`) via `LANDSLIDES_RETURN_YR` (default here 10 kyr; YAML stock 20 kyr) and `cohesion_eff` via `COHESION_EFF` (default here 1000 Pa, tutorial-weak; YAML stock 15000). Use `VERBOSE=False` for ASCII tqdm only. Speed: the bridge reuses HyLands MFD/DepressionFinder scratch objects across steps; elev/soil remap uses **cached barycentric weights** (built once before timesteps; NumPy eval, not per-step LinearND); imports set `LANDLAB_BLAS_THREADS` via `set_blas_threads` for numpy Landlab / DDTD work. BedrockLandslider scar growth stays serial Python; combo-frame `n_jobs` is plot-only (not the sim).




In [ ]:
# yml_b = ROOT / "examples" / "runB_space_hylands.yml"
# with open(ROOT / "examples" / "bridge_runB.yml") as f:
#     cfg = BridgeConfig.from_dict(yaml.safe_load(f))
# # API default raster_scale=1.0; notebook uses 2× finer raster (dx_raster = dx_mesh / scale)
# cfg.raster_scale = 2.0
# cfg.landlab_float32 = bool(LANDLAB_FLOAT32)
# cfg.substeps = landlab_substeps_for_dt(DT_YR, target_sub_yr=LANDLAB_SUB_YR)
# if cfg.mass_wasting is not None:
#     cfg.mass_wasting.kwargs["landslides_return_time"] = float(LANDSLIDES_RETURN_YR)
#     cfg.mass_wasting.kwargs["cohesion_eff"] = float(COHESION_EFF)

# m_b = load_model(yml_b, verbose=VERBOSE, dt=DT_YR, end=END_YR)
# BridgeRunner(m_b, cfg).run()
# print(format_run_done(m_b, label="Run B (SPL+NLD+HyLands bridge)"))
# destroy_model(m_b, verbose=VERBOSE)


## 5. Combo frames (elev | cumED | section) from existing HDF5

Standalone plot cell — **no re-sim**. Reads Run A/B under `examples/outputs/`.

Writes one **3-row** PNG per `PLOT_INTERVAL_YR` step into **`examples/outputs/frames/`** (elev A|B, cumED A|B ± mesh, mid-transect A|B), **200 dpi**, parallel `n_jobs=10`. Layout is tight (narrow gaps / flush colorbars). Elev colorbar: notebook `ELEV_CLIM` (`None` = auto). Optional `ERODEP_CLIM` / `SECTION_ZLIM`. **`SHOW_MESH`**: triangle overlay on the cumED row (`False` = off, default here; set `True` to plot mesh). After export, displays only the frames nearest `PROGRESS_TIMES_YR`.


In [ ]:
# Combo elev colorbar: None = auto shared min/max; tuple = fixed (API default is auto)
ELEV_CLIM = (-150.0, 4000.0)
ERODEP_CLIM = None
SECTION_ZLIM = None
SHOW_MESH = False  # cumED row triangle overlay; True to plot mesh


In [ ]:
# Standalone: combo frames from existing HDF5 only (no BridgeRunner / re-sim).
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "gospl_landlab").exists():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "gospl_landlab").exists():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PLOT_INTERVAL_YR = float(globals().get("PLOT_INTERVAL_YR", 10_000.0))
PROGRESS_TIMES_YR = tuple(
    globals().get("PROGRESS_TIMES_YR", (0.5e6, 2.0e6, 4.0e6))
)
# None = auto (function default); tuple = fixed range for all frames
ELEV_CLIM = globals().get("ELEV_CLIM", (-200.0, 2000.0))
ERODEP_CLIM = globals().get("ERODEP_CLIM", None)
SECTION_ZLIM = globals().get("SECTION_ZLIM", None)
SHOW_MESH = bool(globals().get("SHOW_MESH", False))
DPI = 200
N_JOBS = 10

from gospl_landlab.postprocess import write_combo_frames
from IPython.display import Image, display

out = ROOT / "examples" / "outputs"
paths = write_combo_frames(
    out,
    gif_interval_yr=PLOT_INTERVAL_YR,
    progress_times_yr=PROGRESS_TIMES_YR,
    dpi=DPI,
    n_jobs=N_JOBS,
    elev_clim=None if ELEV_CLIM is None else tuple(ELEV_CLIM),
    erodep_clim=None if ERODEP_CLIM is None else tuple(ERODEP_CLIM),
    zlim=None if SECTION_ZLIM is None else tuple(SECTION_ZLIM),
    show_mesh=SHOW_MESH,
)
print(
    f"wrote {len(paths['combo_frames'])} frames → {paths['frames_dir']} "
    f"(interval={PLOT_INTERVAL_YR/1e3:g} kyr, n_jobs={N_JOBS}, dpi={DPI}, "
    f"elev_clim={paths['elev_clim']})"
)
for p in paths["progress_frames"]:
    print(p)
    display(Image(filename=str(p)))


## 6. Combo MP4 from exported frames

Stitches full-res `examples/outputs/frames/compare_combo_frame_*.png` into **`examples/outputs/compare_combo.mp4`** (H.264, CRF 20, **fps = 16.2** = 1.2× the prior 13.5; no downscale). Runnable alone if frames already exist on disk.



In [ ]:
# Standalone: stitch existing combo PNGs → MP4 (no re-sim / no downscale).
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "gospl_landlab").exists():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "gospl_landlab").exists():
            ROOT = p
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

VIDEO_FPS = 16.2  # 1.2× prior 13.5; do not inherit stale GIF_FPS/VIDEO_FPS from kernel
out = ROOT / "examples" / "outputs"

from gospl_landlab.postprocess import list_combo_frame_pngs, stitch_combo_mp4
from IPython.display import Video, display

frames = None
if "paths" in globals() and paths.get("combo_frames"):
    frames = list(paths["combo_frames"])
if not frames:
    frames = list_combo_frame_pngs(out)
if not frames:
    raise FileNotFoundError(
        f"no compare_combo_frame_*.png under {out / 'frames'} — run the combo frames cell first"
    )
out_mp4 = out / "compare_combo.mp4"  # MP4 stays directly under outputs/
mp4 = stitch_combo_mp4(frames, out_mp4, fps=VIDEO_FPS, crf=20)
print(f"MP4 ({len(frames)} frames, fps={VIDEO_FPS}): {mp4}")
display(Video(filename=str(mp4), embed=True, html_attributes="controls loop"))

